In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
data = {}

for file in RAW_DIR.glob("*.csv"):
    name = file.stem
    data[name] = pd.read_csv(file)

print(data.keys())

dict_keys(['churn_labels', 'content', 'customers', 'customer_feedback', 'payments', 'subscriptions', 'subscription_plans', 'support_tickets', 'viewing_activity'])


In [5]:
for name, df in data.items():
    print(name, df.shape)

churn_labels (8000, 4)
content (500, 9)
customers (8048, 12)
customer_feedback (5046, 6)
payments (110735, 8)
subscriptions (8875, 10)
subscription_plans (3, 6)
support_tickets (6120, 11)
viewing_activity (66422, 9)


In [6]:
# standardizing the column name

In [7]:
for name, df in data.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

In [8]:
for name, df in data.items():
    print(f"\n{name}")
    print(df.columns.tolist())


churn_labels
['customer_id', 'churned', 'churn_date', 'churn_reason']

content
['content_id', 'title', 'content_type', 'genre', 'release_year', 'duration_minutes', 'content_language', 'production_country', 'maturity_rating']

customers
['customer_id', 'first_name', 'last_name', 'age', 'gender', 'country', 'state_or_region', 'city', 'registration_date', 'acquisition_channel', 'customer_segment', 'preferred_language']

customer_feedback
['feedback_id', 'customer_id', 'feedback_date', 'rating', 'feedback_text', 'sentiment_label']

payments
['payment_id', 'customer_id', 'subscription_id', 'payment_date', 'amount', 'payment_method', 'payment_status', 'failed_payment_reason']

subscriptions
['subscription_id', 'customer_id', 'plan_id', 'subscription_start_date', 'subscription_end_date', 'subscription_status', 'auto_renew', 'cancellation_date', 'cancellation_reason', 'monthly_price']

subscription_plans
['plan_id', 'plan_name', 'monthly_price', 'video_quality', 'max_devices', 'advertisements

In [9]:
duplicate_summary = {}

for name, df in data.items():
    duplicate_summary[name] = df.duplicated().sum()

duplicate_summary

{'churn_labels': np.int64(0),
 'content': np.int64(0),
 'customers': np.int64(48),
 'customer_feedback': np.int64(0),
 'payments': np.int64(0),
 'subscriptions': np.int64(0),
 'subscription_plans': np.int64(0),
 'support_tickets': np.int64(0),
 'viewing_activity': np.int64(0)}

In [10]:
# removing duplicates -- exists only in the customers.csv

In [11]:
for name in data:

    data[name] = data[name].drop_duplicates().copy()

In [12]:
# clean customers

In [13]:
# age

In [14]:
customers = data['customers'].copy()

customers['age'] = pd.to_numeric(customers['age'], errors = 'coerce')

invalid_age = (
    (customers['age']< 13) | (customers['age'] > 100)
)

# convert invalid age to nan
customers.loc[invalid_age, "age"] = np.nan

customers["age"] = customers["age"].fillna(
    customers["age"].median()
)

In [15]:
# country

In [16]:
country_mapping = {
    "usa": "united states",
    "us": "united states",
    "u.s.a": "united states",

    "in": "india",
    "bharat": "india",

    "u.k.": "united kingdom",
    "britain": "united kingdom",
    "uk": "united kingdom",

    "fr": "france",
    "ca": "canada",
    "de": "germany",
    "mx": "mexico",

    "uae": "united arab emirates",
    "u.a.e": "united arab emirates",

    "za": "south africa",
    "ng": "nigeria",
    "br": "brazil",
    "jp": "japan",
    "kr": "south korea",
    "s. korea": "south korea",
    "au": "australia",
    "ph": "philippines",
    "ar": "argentina",
    "pl": "poland",
    "id": "indonesia",
    "it": "italy",
    "es": "spain"
}

In [17]:
customers["country"] = (
    customers["country"]
    .str.strip()
    .str.lower()
)

customers["country"] = customers["country"].replace(country_mapping)

customers["country"] = customers["country"].fillna("unknown")

In [18]:
customers["country"].value_counts()

country
united states           1449
india                   1251
united kingdom           613
canada                   419
unknown                  409
brazil                   380
germany                  378
france                   314
mexico                   299
japan                    268
philippines              239
indonesia                237
italy                    225
nigeria                  216
south africa             215
spain                    203
australia                202
south korea              195
argentina                164
poland                   162
united arab emirates     162
Name: count, dtype: int64

### State/Region

The `state_or_region` column contains 100% missing values (8,000 out of 8,000 records). Since no usable information is available in this column and meaningful imputation is not possible, the column was removed from the dataset.

In [19]:
customers = customers.drop(columns=["state_or_region"])

In [20]:
customers.columns

Index(['customer_id', 'first_name', 'last_name', 'age', 'gender', 'country',
       'city', 'registration_date', 'acquisition_channel', 'customer_segment',
       'preferred_language'],
      dtype='str')

In [21]:
# city

In [22]:
# Standardize city names
customers["city"] = (
    customers["city"]
    .str.strip()
    .str.lower()
)

# Handle missing city
customers["city"] = customers["city"].fillna("unknown")

In [23]:
sorted(customers["city"].unique())

['abu dhabi',
 'abuja',
 'ahmedabad',
 'austin',
 'bandung',
 'barcelona',
 'bengaluru',
 'berlin',
 'birmingham',
 'brasilia',
 'brisbane',
 'buenos aires',
 'busan',
 'calgary',
 'cape town',
 'cebu city',
 'chennai',
 'chicago',
 'cologne',
 'cordoba',
 'davao city',
 'delhi',
 'dubai',
 'durban',
 'fortaleza',
 'frankfurt',
 'glasgow',
 'guadalajara',
 'hamburg',
 'houston',
 'hyderabad',
 'ibadan',
 'incheon',
 'jakarta',
 'johannesburg',
 'kano',
 'kolkata',
 'krakow',
 'lagos',
 'leeds',
 'liverpool',
 'london',
 'los angeles',
 'lyon',
 'madrid',
 'manchester',
 'manila',
 'marseille',
 'melbourne',
 'mexico city',
 'miami',
 'milan',
 'monterrey',
 'montreal',
 'mumbai',
 'munich',
 'nagoya',
 'naples',
 'new york',
 'nice',
 'osaka',
 'ottawa',
 'paris',
 'perth',
 'phoenix',
 'pretoria',
 'puebla',
 'pune',
 'rio de janeiro',
 'rome',
 'rosario',
 'salvador',
 'sao paulo',
 'seattle',
 'seoul',
 'seville',
 'sharjah',
 'surabaya',
 'sydney',
 'tokyo',
 'toronto',
 'toulouse'

In [24]:
# age

### Age Cleaning

The `age` column was checked for missing and implausible values. Values outside the defined valid age range were treated as invalid and converted to missing values. Missing age values were then imputed using the median age.

After cleaning, the dataset contains 8,000 valid age values, with ages ranging from 13 to 76.

In [25]:
customers['age'].describe()

count    8000.000000
mean       32.579500
std        10.464206
min        13.000000
25%        25.000000
50%        32.000000
75%        39.000000
max        76.000000
Name: age, dtype: float64

In [26]:
# gender

In [27]:
customers["gender"].value_counts(dropna=False)

gender
Female               3817
Male                 3779
Other                 247
Prefer not to say     157
Name: count, dtype: int64

In [28]:
# first name and last name

In [29]:
customers["first_name"] = customers["first_name"].str.strip()
customers["last_name"] = customers["last_name"].str.strip()

In [30]:
# registration date

In [31]:
customers["registration_date"] = pd.to_datetime(
    customers["registration_date"],
    errors="coerce"
)

### Registration Date

The `registration_date` column contains no missing values and is already stored as a datetime type. The recorded registration dates range from January 1, 2019 to May 31, 2026. No invalid or future dates were identified, so no modification was required.

In [32]:
customers["registration_date"].min()

Timestamp('2019-01-01 00:00:00')

In [33]:
customers["registration_date"].max()

Timestamp('2026-05-31 00:00:00')

In [34]:
# acquistion channel

In [35]:
customers["acquisition_channel"].value_counts(dropna=False)

acquisition_channel
Organic Search              1786
Paid Social                 1427
Referral                    1081
App Store                    990
Affiliate Partner            793
Email Campaign               785
Partner Bundle (Telecom)     640
Direct                       498
Name: count, dtype: int64

In [36]:
customers["acquisition_channel"] = (
    customers["acquisition_channel"]
    .str.strip()
)

In [37]:
customers["acquisition_channel"].unique()

<StringArray>
[          'Organic Search',                 'Referral',
              'Paid Social', 'Partner Bundle (Telecom)',
           'Email Campaign',                'App Store',
                   'Direct',        'Affiliate Partner']
Length: 8, dtype: str

In [38]:
# customer segment

In [39]:
customers['customer_segment'].value_counts(dropna = False)

customer_segment
Individual          4386
Family              1992
Student              952
Corporate Bundle     670
Name: count, dtype: int64

In [40]:
customers['customer_segment'] = (customers['customer_segment'].str.strip())

In [41]:
customers["customer_segment"].unique()

<StringArray>
['Individual', 'Family', 'Student', 'Corporate Bundle']
Length: 4, dtype: str

In [42]:
# preferred lang

In [43]:
customers['preferred_language'].value_counts(dropna = False)

preferred_language
English       2646
Hindi         1082
Spanish        734
Portuguese     533
French         483
German         476
Arabic         422
Korean         405
Japanese       384
Indonesian     286
NaN            251
Tagalog        167
Polish          66
Italian         65
Name: count, dtype: int64

In [44]:
customers['preferred_language'] = (customers['preferred_language'].str.strip())

In [45]:
customers["preferred_language"] = (
    customers["preferred_language"]
    .fillna("Unknown")
)

In [46]:
customers = customers.reset_index(drop=True)

In [47]:
print(customers.info())
print(customers.isna().sum())
print(customers.duplicated().sum())
print(customers['customer_id'].nunique())

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_id          8000 non-null   str           
 1   first_name           8000 non-null   str           
 2   last_name            8000 non-null   str           
 3   age                  8000 non-null   float64       
 4   gender               8000 non-null   str           
 5   country              8000 non-null   str           
 6   city                 8000 non-null   str           
 7   registration_date    8000 non-null   datetime64[us]
 8   acquisition_channel  8000 non-null   str           
 9   customer_segment     8000 non-null   str           
 10  preferred_language   8000 non-null   str           
dtypes: datetime64[us](1), float64(1), str(9)
memory usage: 687.6 KB
None
customer_id            0
first_name             0
last_name              0
age           

In [48]:
customers.to_csv(
    PROCESSED_DIR / "customers.csv",
    index=False
)